# 02. OlmoEarth Linear Probe Training — Native 3m Resolution

**Architecture**: Frozen OlmoEarth ViT encoder + trainable CNN probing head  
**Strategy**: Linear probing — only the ~500K head parameters are trained  
**Resolution**: Native 3m PlanetScope (dynamic 3m→10m inside the model forward pass)  

**Hyperparameter changes from previous run:**
- LR: `1e-4` → `3e-4` (better for randomly-initialized probe head)
- Added: `CosineAnnealingLR` scheduler (prevents oscillation in final epochs)
- Added: `weight_decay=0.01` (AdamW regularization)
- Batch size: `4` → `8` (safe with frozen encoder — no encoder gradients stored)

In [1]:
# ─────────────────────────────────────────────
# CELL 1 — Imports & Environment
# ─────────────────────────────────────────────
import os
import sys
sys.path.append('../')

import torch
from torch.utils.data import DataLoader, random_split

from src.data.dataset import RiverScopeDataset
from src.models.probing import EndToEndOlmoSegmenter
from src.training.trainer import RiverScopeTrainer

# Paths
DATA_ROOT = '../data/raw/RiverScope_dataset'
TRAIN_CSV = os.path.join(DATA_ROOT, 'train.csv')

print('✅ Environment ready.')

✅ Environment ready.


In [2]:
# ─────────────────────────────────────────────
# CELL 2 — Load OlmoEarth Foundation Model
# ─────────────────────────────────────────────
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
# then the rest of your imports below:
import sys
sys.path.append('../')
import torch
...

from huggingface_hub import snapshot_download

from olmoearth_pretrain.model_loader import load_model_from_id

model_repo = 'allenai/OlmoEarth-v1-Base'
print(f'Loading {model_repo}...')

local_dir = snapshot_download(repo_id=model_repo)
foundation_model = load_model_from_id(local_dir, load_weights=True)

print('✅ OlmoEarth-v1-Base loaded.')

Loading allenai/OlmoEarth-v1-Base...


/Users/danny/miniconda3/envs/riverscope_env/lib/python3.11/site-packages/olmoearth_pretrain/model_loader.py:38: UserWarning: olmo-core not installed. Running in inference-only mode. For training: pip install olmoearth-pretrain[training]
  from olmoearth_pretrain.config import Config


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/Users/danny/miniconda3/envs/riverscope_env/lib/python3.11/importlib/__init__.py:126: FutureWarning: The 'helios' package has been renamed to 'olmoearth_pretrain'. Please update your imports; this compatibility shim will be removed in a future release.
  return _bootstrap._gcd_import(name[level:], package, level)


✅ OlmoEarth-v1-Base loaded.


In [3]:
# ─────────────────────────────────────────────
# CELL 3 — Dataset & DataLoaders
# ─────────────────────────────────────────────
BATCH_SIZE = 8   # Safe with frozen encoder — no encoder gradients stored

full_dataset = RiverScopeDataset(TRAIN_CSV, DATA_ROOT)

# Reproducible 80/20 split — same seed as baseline for fair comparison
train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'✅ Dataset: {len(train_dataset)} train | {len(val_dataset)} val | batch_size={BATCH_SIZE}')

✅ Dataset: 629 train | 158 val | batch_size=8


In [4]:
# ─────────────────────────────────────────────
# CELL 4 — Build EndToEndOlmoSegmenter
# ─────────────────────────────────────────────
# EndToEndOlmoSegmenter wraps:
#   1. Frozen OlmoEarth ViT encoder (no gradients)
#   2. Trainable CNN probing head (14x14 -> 224x224)
# Dynamic resolution: 3m input -> squeeze to 10m inside forward() -> upsample back to 3m
model = EndToEndOlmoSegmenter(
    foundation_model=foundation_model,
    embed_dim=768,
    num_classes=1
)

# Verify only the probe head is trainable
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.1f}% — probe head only)')
print(f'Frozen params:    {total_params - trainable_params:,} (OlmoEarth encoder)')

Total params:     209,372,865
Trainable params: 1,917,633 (0.9% — probe head only)
Frozen params:    207,455,232 (OlmoEarth encoder)


In [5]:
# ─────────────────────────────────────────────
# CELL 5 — Configure Trainer & Run
# ─────────────────────────────────────────────
# Hyperparameter rationale:
#   lr=3e-4:      Higher than full fine-tuning because probe is randomly initialized
#                 and the frozen encoder outputs are stable features.
#   CosineAnnealingLR: Decays lr -> 1e-6 over 25 epochs. Prevents oscillation
#                 around the loss minimum in final epochs (+1-2 IoU vs flat lr).
#   weight_decay: 0.01 via AdamW to regularize ~500K probe params.
#   pos_weight=20: Calibrated to 4.6% river pixel rate: (1-0.046)/0.046 ≈ 20.7
EPOCHS     = 25
SAVE_PATH  = 'best_olmo_cosine_probe.pth'

trainer = RiverScopeTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    lr=3e-4,
    epochs=EPOCHS,
)

print('🚀 Starting OlmoEarth Linear Probe Training (Cosine LR Schedule)...')
print(f'   Target: beat previous best of 0.5155 Native IoU\n')

best_iou = trainer.fit(epochs=EPOCHS, save_path=SAVE_PATH)

print(f'\n✅ Training complete! Max Native Validation IoU: {best_iou:.4f}')
print(f'   Previous best (flat lr=1e-4): 0.5155')
print(f'   Improvement: {best_iou - 0.5155:+.4f}')

🚀 Starting OlmoEarth Linear Probe Training (Cosine LR Schedule)...
   Target: beat previous best of 0.5155 Native IoU

Starting Training on device: mps...
LR schedule: CosineAnnealing | Start LR: 3.0e-04 -> eta_min: 1e-6 over 25 epochs


[Epoch 1/25] | LR: 3.00e-04


Training:   0%|          | 0/79 [00:00<?, ?it/s]


NotImplementedError: The operator 'aten::_upsample_bicubic2d_aa.out' is not currently implemented for the MPS device. If you want this op to be considered for addition please comment on https://github.com/pytorch/pytorch/issues/141287 and mention use-case, that resulted in missing op as well as commit hash Unknown. As a temporary fix, you can set the environment variable `PYTORCH_ENABLE_MPS_FALLBACK=1` to use the CPU as a fallback for this op. WARNING: this will be slower than running natively on MPS.

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Visual Inference
# ─────────────────────────────────────────────
import matplotlib.pyplot as plt

device = trainer.device

# Load best checkpoint
model.load_state_dict(torch.load(SAVE_PATH, map_location=device, weights_only=True))
model.eval()

# Sample 3 random tiles from the val set
vis_loader = DataLoader(val_dataset, batch_size=3, shuffle=True)
images, masks, _ = next(iter(vis_loader))
images, masks = images.to(device), masks.to(device)

from torch.nn.attention import sdpa_kernel, SDPBackend
with torch.no_grad(), sdpa_kernel(SDPBackend.MATH):
    logits = model(images)

predictions = (torch.sigmoid(logits) > 0.5).float()

images_cpu = images.cpu()
masks_cpu  = masks.cpu().squeeze()
preds_cpu  = predictions.cpu().squeeze()

fig, axs = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle(f'OlmoEarth Linear Probe — Cosine Schedule | Best IoU: {best_iou:.4f}', fontsize=14)

for i in range(3):
    # RGB: B04 (Red, idx 3), B03 (Green, idx 2), B02 (Blue, idx 1)
    rgb = images_cpu[i, 0, [3, 2, 1], :, :].permute(1, 2, 0).numpy()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)

    axs[i, 0].imshow(rgb)
    axs[i, 0].set_title('Input RGB (PlanetScope/S2)')
    axs[i, 0].axis('off')

    axs[i, 1].imshow(masks_cpu[i], cmap='Blues')
    axs[i, 1].set_title('Ground Truth Mask')
    axs[i, 1].axis('off')

    axs[i, 2].imshow(preds_cpu[i], cmap='Blues')
    axs[i, 2].set_title('OlmoEarth Prediction (Cosine)')
    axs[i, 2].axis('off')

plt.tight_layout()
plt.savefig('olmo_cosine_inference.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualization saved to olmo_cosine_inference.png')